# Filtrado probabilístico

Compara Bloom Filter y Count-Min Sketch con el conjunto y el conteo exactos. Ejecuta todas las celdas. Este notebook contiene su código y no usa los archivos de `test/`.


In [ ]:
RUN_MODE = "quick"  # "quick" mide 1K, 10K y 100K. "full" añade 1M y 10M.
SEED = 42
DATASET_FILE = ""  # Ruta a 2019-Oct.csv. Vacío: muestra sintética solo para validar.

print("Modo:", RUN_MODE)
print("Semilla:", SEED)
print("CSV:", DATASET_FILE or "sintético, solo validación")


In [ ]:
import platform
import time
import tracemalloc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "01_filtrado_probabilistico.ipynb").exists():
    ROOT = ROOT / "pruebas_algoritmos"

def sizes(mode):
    return [1_000, 10_000, 100_000] if mode == "quick" else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

def repetitions(n, mode):
    if mode == "quick":
        return 7
    return 3 if n >= 1_000_000 else 100

def prepare_output(folder):
    results, figures = folder / "resultados", folder / "figuras"
    results.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return results, figures

def resolve_data(data):
    if not data:
        return None
    candidate = Path(data)
    options = [candidate, Path.cwd() / candidate, ROOT / candidate, ROOT.parent / candidate]
    found = next((option for option in options if option.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"No existe el CSV indicado: {data}")
    return found

def load_events(data, n, seed):
    found = resolve_data(data)
    if found:
        cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "brand", "category_code"]
        frame = pd.read_csv(found, nrows=n, usecols=lambda column: column in cols)
        return frame, "REES46 CSV"
    rng = np.random.default_rng(seed)
    products = max(300, min(n // 8, 20_000))
    users = max(200, min(n // 5, 50_000))
    frame = pd.DataFrame({
        "event_time": pd.date_range("2019-10-01", periods=n, freq="s"),
        "event_type": rng.choice(["view", "cart", "purchase"], size=n, p=[0.83, 0.12, 0.05]),
        "product_id": rng.integers(1_000_000, 1_000_000 + products, size=n),
        "price": np.round(rng.lognormal(4.2, 0.75, n), 2),
        "user_id": rng.integers(1, users, size=n),
        "user_session": rng.integers(1, max(20, n // 4), size=n).astype(str),
        "brand": rng.choice(["samsung", "apple", "sony", "unknown"], size=n),
        "category_code": rng.choice(["electronics.phone", "appliances.kitchen", "computers"], size=n),
    })
    return frame, "SINTÉTICO (solo validación; no usar en el informe)"

def measure(action, reps):
    timings = []
    for _ in range(reps):
        start = time.perf_counter()
        action()
        timings.append((time.perf_counter() - start) * 1_000)
    tracemalloc.start()
    action()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(np.mean(timings)), float(np.std(timings)), peak / 1024 ** 2

def record(rows, **fields):
    rows.append({
        "experimento": fields["experiment"], "algoritmo": fields["algorithm"], "n": fields["n"],
        "metrica": fields["metric"], "valor": fields["value"], "tiempo_promedio_ms": fields["time_ms"],
        "desviacion_ms": fields["std_ms"], "memoria_pico_mb": fields["memory_mb"],
        "repeticiones": fields["reps"], "semilla": fields["seed"],
        "plataforma": platform.platform(), "fuente": fields["source"],
    })

def save(rows, name, output):
    results, _ = prepare_output(output)
    path = results / f"{name}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Tabla: {path}")
    return path

def line_chart(frame, title, filename, output, value="tiempo_promedio_ms", log=True):
    _, figures = prepare_output(output)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for algorithm, group in frame.groupby("algoritmo"):
        group = group.sort_values("n")
        if log:
            group = group[group[value] > 0]
        if group.empty:
            continue
        ax.plot(group["n"], group[value], marker="o", label=algorithm)
    ax.set(title=title, xlabel="Elementos", ylabel=value.replace("_", " "))
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = figures / filename
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    print(f"Figura: {path}")
    return path

import collections
import hashlib
import math
def stable_hash(value: object, salt: int = 0) -> int:
    return int.from_bytes(hashlib.blake2b(f"{salt}|{value}".encode(), digest_size=8).digest(), "little")


class BloomFilter:
    def __init__(self, capacity: int, error_rate: float = 0.01):
        self.bits_count = max(8, int(-capacity * math.log(error_rate) / math.log(2) ** 2))
        self.hashes = max(1, round(self.bits_count / max(capacity, 1) * math.log(2)))
        self.bits = bytearray((self.bits_count + 7) // 8)

    def _locations(self, value: object):
        for salt in range(self.hashes):
            yield stable_hash(value, salt) % self.bits_count

    def add(self, value: object) -> None:
        for pos in self._locations(value):
            self.bits[pos // 8] |= 1 << (pos % 8)

    def contains(self, value: object) -> bool:
        return all(self.bits[pos // 8] & (1 << (pos % 8)) for pos in self._locations(value))


class CountMinSketch:
    def __init__(self, width: int = 2048, depth: int = 5):
        self.width, self.depth = width, depth
        self.table = np.zeros((depth, width), dtype=np.uint32)

    def add(self, value: object, count: int = 1) -> None:
        for row in range(self.depth):
            self.table[row, stable_hash(value, row) % self.width] += count

    def estimate(self, value: object) -> int:
        return int(min(self.table[row, stable_hash(value, row) % self.width] for row in range(self.depth)))

def run(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    max_n = max(sizes(mode))
    df, source = load_events(data, max_n, seed)
    values = df["user_id"].astype(int).to_numpy()
    for n in sizes(mode):
        sample, reps = values[:n], repetitions(n, mode)
        inserted = set(sample[::10])
        negatives = [int(v) for v in sample if v not in inserted][:10_000]
        bloom = BloomFilter(len(inserted))
        bloom_time, bloom_std, bloom_mem = measure(lambda: [bloom.add(int(v)) for v in inserted], reps)
        for value in inserted:
            bloom.add(value)
        fp = sum(bloom.contains(v) for v in negatives) / max(1, len(negatives))
        record(rows, experiment="Bloom vs set", algorithm="Bloom Filter", n=n, metric="falsos_positivos",
               value=fp, time_ms=bloom_time, std_ms=bloom_std, memory_mb=bloom_mem, reps=reps, seed=seed, source=source)
        exact_time, exact_std, exact_mem = measure(lambda: set(inserted), reps)
        record(rows, experiment="Bloom vs set", algorithm="set exacto", n=n, metric="falsos_positivos",
               value=0, time_ms=exact_time, std_ms=exact_std, memory_mb=exact_mem, reps=reps, seed=seed, source=source)

        cms = CountMinSketch()
        cms_time, cms_std, cms_mem = measure(lambda: [cms.add(int(v)) for v in sample], reps)
        for value in sample:
            cms.add(int(value))
        exact = collections.Counter(map(int, sample))
        mae = sum(abs(cms.estimate(key) - count) for key, count in exact.items()) / max(1, len(exact))
        record(rows, experiment="CMS vs Counter", algorithm="Count-Min Sketch", n=n, metric="mae_frecuencia",
               value=mae, time_ms=cms_time, std_ms=cms_std, memory_mb=cms_mem, reps=reps, seed=seed, source=source)
        counter_time, counter_std, counter_mem = measure(lambda: collections.Counter(map(int, sample)), reps)
        record(rows, experiment="CMS vs Counter", algorithm="Counter exacto", n=n, metric="mae_frecuencia",
               value=0, time_ms=counter_time, std_ms=counter_std, memory_mb=counter_mem, reps=reps, seed=seed, source=source)
    path = save(rows, "probabilisticos", out)
    result = pd.read_csv(path)
    line_chart(result[result["experimento"] == "Bloom vs set"], "Bloom Filter vs set: tiempo", "bloom_tiempo.png", out)
    line_chart(result[result["experimento"] == "Bloom vs set"], "Bloom Filter: falsos positivos", "bloom_calidad.png", out, value="valor", log=False)
    line_chart(result[result["experimento"] == "CMS vs Counter"], "Count-Min vs Counter: tiempo", "cms_tiempo.png", out)
    line_chart(result[result["experimento"] == "CMS vs Counter"], "Count-Min: error de frecuencia", "cms_calidad.png", out, value="valor", log=False)
    return result

resultado = run(mode=RUN_MODE, data=DATASET_FILE or None, seed=SEED)
resultado


## Lectura

El pipeline usa Bloom y Count-Min como señales rápidas y conserva el conjunto exacto antes de bloquear una compra. La figura de calidad muestra falsos positivos y error de frecuencia. El conjunto exacto no se descarta; se reserva para la decisión crítica.
